# 19. Supervised Learning: Linear Discriminant Analysis (LDA)

## Algorithm Category
**Type**: Supervised Learning - Classification & Dimensionality Reduction  
**Complexity**: Medium  
**Use Case**: Classification with dimensionality reduction using linear projections

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of LDA
- Implement LDA for both classification and dimensionality reduction
- Understand the difference between LDA and PCA
- Visualize LDA projections and decision boundaries
- Apply LDA to real-world classification problems

## Historical Context

Linear Discriminant Analysis was developed by Ronald Fisher in 1936:
- Fisher, R.A. (1936): "The use of multiple measurements in taxonomic problems"
- Originally developed for classification, later adapted for dimensionality reduction
- Foundation for many modern classification techniques

**Key Papers/References:**
- Fisher, R.A. (1936). "The use of multiple measurements in taxonomic problems"
- Belhumeur, P.N., et al. (1997). "Eigenfaces vs. Fisherfaces"

## When to Use Linear Discriminant Analysis

LDA is appropriate when:
- You need both classification and dimensionality reduction
- Data is approximately normally distributed
- Classes have similar covariance matrices
- You want to maximize class separability
- Working with high-dimensional data
- Interpretability of projections is important

## Theory & Mechanics

### Mathematical Foundation

LDA finds linear combinations of features that maximize class separability.

**Fisher's Linear Discriminant:**
$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

Where:
- $S_B$: Between-class scatter matrix
- $S_W$: Within-class scatter matrix

**Between-class scatter:**
$$S_B = \sum_{i=1}^{c} n_i (\mu_i - \mu)(\mu_i - \mu)^T$$

**Within-class scatter:**
$$S_W = \sum_{i=1}^{c} \sum_{x \in C_i} (x - \mu_i)(x - \mu_i)^T$$

**Solution:**
$$w = S_W^{-1}(\mu_1 - \mu_2)$$ (for 2 classes)

**Generalized Eigenvalue Problem:**
$$S_B w = \lambda S_W w$$

### How It Works

1. **Calculate class means**: $\mu_i$ for each class
2. **Calculate scatter matrices**: $S_B$ (between-class) and $S_W$ (within-class)
3. **Solve eigenvalue problem**: Find eigenvectors of $S_W^{-1} S_B$
4. **Project data**: Transform to lower-dimensional space
5. **Classify**: Use projected features for classification

### Key Assumptions

1. **Normality**: Features are normally distributed
2. **Equal covariance**: All classes have same covariance matrix
3. **Independence**: Observations are independent

### Limitations

- Assumes normal distribution (may not hold in practice)
- Assumes equal covariance matrices (homoscedasticity)
- Limited to linear decision boundaries
- Number of components limited by (number of classes - 1)


## Implementation

Let's implement LDA for both classification and dimensionality reduction.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_iris,  # Iris flower classification dataset
    load_wine  # Wine classification dataset
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # LDA for classification and dimensionality reduction
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score  # Cross-validation scoring
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy
    classification_report  # Detailed classification metrics
)
from sklearn.preprocessing import StandardScaler  # Feature scaling

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_classifier  # Supervised learning utilities
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix  # Visualize confusion matrix
)
from src.processing.preprocessing import scale_features  # Normalize features
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# load_iris() loads the Iris flower dataset from scikit-learn
# This is a multiclass classification problem: predict flower species from measurements
iris = load_iris()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Flower measurements
# iris.data contains feature values (150 samples × 4 features)
# We convert to DataFrame for easier manipulation
X = pd.DataFrame(iris.data, columns=iris.feature_names)
# Features: sepal length, sepal width, petal length, petal width (4 measurements)

# y = Target (output): Flower species (what we want to predict)
# iris.target contains class labels (0, 1, or 2 for each sample)
y = pd.Series(iris.target, name='Species')
# 0 = setosa, 1 = versicolor, 2 = virginica

print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Classes: {iris.target_names.tolist()}")  # Output: ['setosa', 'versicolor', 'virginica']
print(f"Class distribution:\n{y.value_counts()}")  # Shows: 50 samples per class (balanced dataset)

# ============================================
# FEATURE SCALING: Important for LDA
# ============================================

# LDA assumes features are normally distributed and have similar scales
# Scaling ensures all features contribute equally to the analysis
# scale_features() normalizes features to have mean=0 and std=1
X_scaled, scaler = scale_features(X, fit=True)
# X_scaled: Features normalized (mean=0, std=1 for each column)
# scaler: The scaling object (needed to scale test data with same transformation)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# X_train: 120 samples for training (scaled features)
# X_test: 30 samples for testing
# y_train: Labels for training samples
# y_test: Labels for test samples (ground truth)


In [ ]:
# ============================================
# MODEL CREATION: Linear Discriminant Analysis (LDA)
# ============================================

# Create LinearDiscriminantAnalysis model
# LDA can be used for both classification and dimensionality reduction
# It finds linear combinations of features that maximize class separability
lda_classifier = LinearDiscriminantAnalysis()
# Default parameters work well for most cases
# LDA automatically determines number of components (max = n_classes - 1)

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the LDA model
# The algorithm:
# 1. Calculates class means (centroid of each class)
# 2. Calculates within-class scatter matrix (how spread out each class is)
# 3. Calculates between-class scatter matrix (how separated classes are)
# 4. Finds linear projections that maximize class separability
# 5. Uses these projections for classification
lda_classifier.fit(X_train, y_train)  # Train the model

print("LDA Classifier:")
# Number of components: LDA can reduce dimensions
# Maximum number of components = number of classes - 1
# For 3 classes, max components = 2
print(f"Number of components: {lda_classifier.n_components_}")  # Usually 2 for 3 classes

# Explained variance ratio: How much variance each component explains
# Higher = component captures more information
print(f"Explained variance ratio: {lda_classifier.explained_variance_ratio_}")
# Example: [0.99, 0.01] means first component explains 99% of variance

# ============================================
# MAKING PREDICTIONS
# ============================================

# .predict() makes predictions using the trained LDA model
# LDA uses the learned linear projections to classify new samples
y_pred = lda_classifier.predict(X_test)  # Class predictions (0, 1, or 2)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"\nClassification Results:")
print(f"  Test Accuracy: {accuracy:.3f}")  # Display accuracy

# ============================================
# EVALUATING MODEL PERFORMANCE
# ============================================

# Evaluate with helper function
results = evaluate_classifier(lda_classifier, X_test, y_test)
# Returns dictionary with accuracy and other metrics

# Calculate detailed classification metrics
metrics = calculate_classification_metrics(y_test.values, y_pred)
# Returns dictionary with: accuracy, precision, recall, F1

print(f"  Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")
# Precision: Of predicted positives, how many were actually positive
# Recall: Of actual positives, how many did we catch
# F1: Harmonic mean of precision and recall (balances both)


In [ ]:
# LDA for Dimensionality Reduction
lda_reducer = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda_reducer.fit_transform(X_scaled, y)

print(f"\nDimensionality Reduction:")
print(f"  Original shape: {X_scaled.shape}")
print(f"  Reduced shape: {X_lda.shape}")
print(f"  Explained variance ratio: {lda_reducer.explained_variance_ratio_}")

# Visualize LDA projection
plt.figure(figsize=(10, 6))
colors = ['red', 'green', 'blue']
for idx, (target_name, color) in enumerate(zip(iris.target_names, colors)):
    plt.scatter(X_lda[y == idx, 0], X_lda[y == idx, 1], 
               c=color, label=target_name, alpha=0.7, edgecolors='black')
plt.xlabel('First Linear Discriminant')
plt.ylabel('Second Linear Discriminant')
plt.title('LDA Projection of Iris Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the model and compare with PCA.


In [ ]:
# Validation 1: Cross-validation
cv_scores = cross_val_score(lda_classifier, X_scaled, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 2: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > 0.5, "Accuracy should be better than random!"
print("\n✓ Validation checks passed")


In [ ]:
# Compare LDA with PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Train classifier on PCA-reduced data
from sklearn.linear_model import LogisticRegression
lr_pca = LogisticRegression(max_iter=1000, random_state=42)
lr_lda = LogisticRegression(max_iter=1000, random_state=42)

X_pca_train, X_pca_test, _, _ = train_test_split(X_pca, y, test_size=0.2, random_state=42)
X_lda_train, X_lda_test, _, _ = train_test_split(X_lda, y, test_size=0.2, random_state=42)

lr_pca.fit(X_pca_train, y_train)
lr_lda.fit(X_lda_train, y_train)

pca_acc = accuracy_score(y_test, lr_pca.predict(X_pca_test))
lda_acc = accuracy_score(y_test, lr_lda.predict(X_lda_test))

print("Comparison: PCA vs LDA")
print(f"  PCA + Logistic Regression: {pca_acc:.3f}")
print(f"  LDA + Logistic Regression: {lda_acc:.3f}")
print(f"  Direct LDA Classifier: {accuracy:.3f}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA plot
for idx, (target_name, color) in enumerate(zip(iris.target_names, colors)):
    axes[0].scatter(X_pca[y == idx, 0], X_pca[y == idx, 1], 
                   c=color, label=target_name, alpha=0.7, edgecolors='black')
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
axes[0].set_title('PCA Projection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LDA plot
for idx, (target_name, color) in enumerate(zip(iris.target_names, colors)):
    axes[1].scatter(X_lda[y == idx, 0], X_lda[y == idx, 1], 
                   c=color, label=target_name, alpha=0.7, edgecolors='black')
axes[1].set_xlabel('First Linear Discriminant')
axes[1].set_ylabel('Second Linear Discriminant')
axes[1].set_title('LDA Projection')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Real-World Application

Let's apply LDA to a more complex dataset and visualize decision boundaries.


In [ ]:
# Wine dataset (more classes and features)
wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine = pd.Series(wine.target, name='Class')

X_wine_scaled, _ = scale_features(X_wine, fit=True)
X_w_train, X_w_test, y_w_train, y_w_test = split_data(X_wine_scaled, y_wine, test_size=0.2, random_state=42)

# LDA on Wine dataset
lda_wine = LinearDiscriminantAnalysis()
lda_wine.fit(X_w_train, y_w_train)

y_w_pred = lda_wine.predict(X_w_test)
wine_accuracy = accuracy_score(y_w_test, y_w_pred)

print("Wine Dataset Results:")
print(f"  Original features: {X_wine.shape[1]}")
print(f"  LDA components: {lda_wine.n_components_}")
print(f"  Test Accuracy: {wine_accuracy:.3f}")
print(f"  Explained variance: {lda_wine.explained_variance_ratio_}")

# Reduce to 2D for visualization
lda_wine_2d = LinearDiscriminantAnalysis(n_components=2)
X_wine_lda = lda_wine_2d.fit_transform(X_wine_scaled, y_wine)

plt.figure(figsize=(10, 6))
for idx in range(len(wine.target_names)):
    plt.scatter(X_wine_lda[y_wine == idx, 0], X_wine_lda[y_wine == idx, 1],
               label=wine.target_names[idx], alpha=0.7, edgecolors='black')
plt.xlabel('First Linear Discriminant')
plt.ylabel('Second Linear Discriminant')
plt.title('LDA Projection of Wine Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **LDA Basics**
   - Supervised dimensionality reduction (uses class labels)
   - Maximizes class separability
   - Can be used for both classification and dimensionality reduction

2. **Mathematical Foundation**
   - Fisher's linear discriminant criterion
   - Between-class vs within-class scatter
   - Generalized eigenvalue problem

3. **Key Differences from PCA**
   - **LDA**: Supervised, maximizes class separation
   - **PCA**: Unsupervised, maximizes variance
   - **LDA**: Limited to (n_classes - 1) components
   - **PCA**: Can have up to n_features components

4. **Best Practices**
   - Scale features before applying LDA
   - Check assumptions (normality, equal covariance)
   - Use for dimensionality reduction before classification
   - Compare with PCA for unsupervised reduction

### When to Use Linear Discriminant Analysis

✅ **Good for:**
- Classification with dimensionality reduction
- When class labels are available
- High-dimensional classification problems
- When you need interpretable projections
- Preprocessing step before other classifiers

❌ **Not ideal for:**
- Non-normal data (assumptions violated)
- Unequal covariance matrices
- Non-linear class boundaries
- When you need more than (n_classes - 1) components
- Very large datasets (computational cost)

### Next Steps

- Try **Quadratic Discriminant Analysis (QDA)** for non-linear boundaries
- Explore **Regularized LDA** for high-dimensional data
- Compare with **PCA** for dimensionality reduction
- Use LDA as preprocessing for other classifiers
